# Credit Card Fraud Detection - Data Exploration

This notebook explores the credit card fraud dataset to understand patterns, distributions, and potential fraud indicators.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load data
df = pd.read_csv('../data/raw/creditcard.csv')
print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df['Class'].mean():.4f}")

## Basic Dataset Information

In [ ]:
# Dataset info
print("Dataset Info:")
print(df.info())

print("\nBasic Statistics:")
print(df.describe())

print("\nClass Distribution:")
print(df['Class'].value_counts())
print(f"Fraud percentage: {df['Class'].mean() * 100:.3f}%")

## Fraud Distribution Analysis

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Count plot
df['Class'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Class Distribution (Count)')
axes[0].set_xlabel('Class (0=Normal, 1=Fraud)')
axes[0].set_ylabel('Count')

# Percentage plot
(df['Class'].value_counts() / len(df) * 100).plot(kind='bar', ax=axes[1])
axes[1].set_title('Class Distribution (Percentage)')
axes[1].set_xlabel('Class (0=Normal, 1=Fraud)')
axes[1].set_ylabel('Percentage')

plt.tight_layout()
plt.show()

## Transaction Amount Analysis

In [ ]:
# Amount distribution by class
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Overall amount distribution
df['Amount'].hist(bins=50, ax=axes[0,0], alpha=0.7)
axes[0,0].set_title('Overall Amount Distribution')
axes[0,0].set_xlabel('Amount')
axes[0,0].set_ylabel('Frequency')

# Amount distribution by class
df[df['Class']==0]['Amount'].hist(bins=50, alpha=0.7, label='Normal', ax=axes[0,1])
df[df['Class']==1]['Amount'].hist(bins=50, alpha=0.7, label='Fraud', ax=axes[0,1])
axes[0,1].set_title('Amount Distribution by Class')
axes[0,1].set_xlabel('Amount')
axes[0,1].set_ylabel('Frequency')
axes[0,1].legend()

# Log scale amount distribution
df['Amount_log'] = np.log1p(df['Amount'])
df[df['Class']==0]['Amount_log'].hist(bins=50, alpha=0.7, label='Normal', ax=axes[1,0])
df[df['Class']==1]['Amount_log'].hist(bins=50, alpha=0.7, label='Fraud', ax=axes[1,0])
axes[1,0].set_title('Log Amount Distribution by Class')
axes[1,0].set_xlabel('Log(Amount + 1)')
axes[1,0].set_ylabel('Frequency')
axes[1,0].legend()

# Box plot
df.boxplot(column='Amount', by='Class', ax=axes[1,1])
axes[1,1].set_title('Amount Distribution by Class (Box Plot)')
axes[1,1].set_xlabel('Class')
axes[1,1].set_ylabel('Amount')

plt.tight_layout()
plt.show()

# Amount statistics by class
print("Amount Statistics by Class:")
print(df.groupby('Class')['Amount'].describe())

## Time-based Analysis

In [ ]:
# Time-based patterns
df['Time_hours'] = df['Time'] / 3600
df['Hour'] = (df['Time'] / 3600) % 24

# Fraud rate by hour
hourly_fraud = df.groupby(df['Hour'].astype(int))['Class'].agg(['count', 'sum', 'mean']).reset_index()
hourly_fraud.columns = ['Hour', 'Total_Transactions', 'Fraud_Count', 'Fraud_Rate']

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Transaction volume by hour
axes[0].bar(hourly_fraud['Hour'], hourly_fraud['Total_Transactions'], alpha=0.7)
axes[0].set_title('Transaction Volume by Hour of Day')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Transactions')

# Fraud rate by hour
axes[1].plot(hourly_fraud['Hour'], hourly_fraud['Fraud_Rate'], marker='o', color='red')
axes[1].set_title('Fraud Rate by Hour of Day')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Rate')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("Hourly Fraud Statistics:")
print(hourly_fraud.sort_values('Fraud_Rate', ascending=False).head(10))

## PCA Features Analysis

In [ ]:
# PCA features correlation with fraud
pca_features = [f'V{i}' for i in range(1, 29)]
fraud_correlation = df[pca_features + ['Class']].corr()['Class'].abs().sort_values(ascending=False)

plt.figure(figsize=(12, 8))
fraud_correlation[1:-1].plot(kind='bar')  # Exclude 'Class' itself
plt.title('PCA Features Correlation with Fraud (Absolute Values)')
plt.xlabel('PCA Features')
plt.ylabel('Absolute Correlation with Fraud')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Top 10 PCA features correlated with fraud:")
print(fraud_correlation[1:11])  # Top 10 excluding Class itself

## Interactive Visualization with Plotly

In [ ]:
# Interactive scatter plot
sample_df = df.sample(n=5000, random_state=42)  # Sample for performance

fig = px.scatter(
    sample_df, 
    x='Time_hours', 
    y='Amount', 
    color='Class',
    title='Transaction Patterns: Time vs Amount',
    labels={'Class': 'Transaction Type', 'Time_hours': 'Time (Hours)', 'Amount': 'Amount'},
    color_discrete_map={0: 'blue', 1: 'red'}
)

fig.update_layout(height=600)
fig.show()

In [ ]:
# Interactive correlation heatmap
correlation_matrix = df[['Time', 'Amount'] + pca_features[:10] + ['Class']].corr()

fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmid=0
))

fig.update_layout(
    title='Feature Correlation Matrix',
    xaxis_title='Features',
    yaxis_title='Features',
    height=600
)

fig.show()

## Key Insights

1. **Severe Class Imbalance**: Only 0.17% of transactions are fraudulent
2. **Amount Patterns**: Fraudulent transactions tend to have different amount distributions
3. **Time Patterns**: Fraud patterns may vary by hour of day
4. **PCA Features**: Several V features show correlation with fraud
5. **Feature Engineering Opportunities**: Time-based and amount-based features could be valuable

## Next Steps
- Engineer time-based features (hour of day, day of week)
- Create amount-based features (percentiles, categories)
- Handle class imbalance with appropriate techniques
- Build and compare multiple models